In [19]:
import fiftyone as fo
print(fo.__version__)

import fiftyone.zoo as foz
from fiftyone import ViewField as F
import fiftyone.brain as fob

import sys
from pathlib import Path

1.13.4


In [27]:
root_dir = str(Path().resolve().parent) 

if root_dir not in sys.path:
    sys.path.insert(0, root_dir)

In [28]:
sys.path

['/home/felipe/Documentos/Projetos/pothole-detection',
 '/home/felipe/miniconda3/envs/pothole-detection/lib/python310.zip',
 '/home/felipe/miniconda3/envs/pothole-detection/lib/python3.10',
 '/home/felipe/miniconda3/envs/pothole-detection/lib/python3.10/lib-dynload',
 '',
 '/home/felipe/miniconda3/envs/pothole-detection/lib/python3.10/site-packages',
 '/home/felipe/miniconda3/envs/pothole-detection/lib/python3.10/site-packages/setuptools/_vendor']

In [21]:
Path.cwd().parent

PosixPath('/home/felipe/Documentos/Projetos/pothole-detection')

In [34]:
dataset_dir = Path.cwd().parent / "data/Pothole.v1-raw.yolo26/"
yaml_dir = Path.cwd().parent / "data/Pothole.v1-raw.yolo26/data.yaml"
print(dataset_dir)
print(yaml_dir)

/home/felipe/Documentos/Projetos/pothole-detection/data/Pothole.v1-raw.yolo26
/home/felipe/Documentos/Projetos/pothole-detection/data/Pothole.v1-raw.yolo26/data.yaml


In [35]:
fo.list_datasets()

['pothole-yolo26', 'quickstart']

In [41]:
fo.delete_dataset('pothole-yolo26')

In [42]:
dataset_train = fo.Dataset.from_dir(
    dataset_dir='../data/Pothole.v1-raw.yolo26/',
    dataset_type=fo.types.YOLOv5Dataset,
    split="train",
    name="pothole-yolo26",
    yaml_path = yaml_dir
)

 100% |█████████████████| 465/465 [513.9ms elapsed, 0s remaining, 906.9 samples/s]     


In [43]:
dataset_train

Name:        pothole-yolo26
Media type:  image
Num samples: 465
Persistent:  False
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    ground_truth:     fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)

In [44]:
clip = foz.load_zoo_model("clip-vit-base32-torch")

In [45]:
dataset_train.compute_embeddings(
    clip,
    embeddings_field="clip_embeddings"
)

 100% |█████████████████| 465/465 [2.8s elapsed, 0s remaining, 205.5 samples/s]      


In [46]:
fob.compute_visualization(
    dataset_train,
    embeddings="clip_embeddings",
    method="tsne",
    brain_key="clip_tsne"
)

dataset_train

Generating visualization...
[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 465 samples in 0.000s...
[t-SNE] Computed neighbors for 465 samples in 0.135s...
[t-SNE] Computed conditional probabilities for sample 465 / 465
[t-SNE] Mean sigma: 1.788479
[t-SNE] Computed conditional probabilities in 0.006s
[t-SNE] Iteration 50: error = 64.4143600, gradient norm = 0.2338395 (50 iterations in 0.021s)
[t-SNE] Iteration 100: error = 65.3562927, gradient norm = 0.2712742 (50 iterations in 0.021s)
[t-SNE] Iteration 150: error = 66.0752945, gradient norm = 0.2400546 (50 iterations in 0.022s)
[t-SNE] Iteration 200: error = 64.3927612, gradient norm = 0.2688973 (50 iterations in 0.023s)
[t-SNE] Iteration 250: error = 65.1978836, gradient norm = 0.2155748 (50 iterations in 0.022s)
[t-SNE] KL divergence after 250 iterations with early exaggeration: 65.197884
[t-SNE] Iteration 300: error = 0.9853454, gradient norm = 0.0045628 (50 iterations in 0.018s)
[t-SNE] Iteration 350: error = 0.9288551,

Name:        pothole-yolo26
Media type:  image
Num samples: 465
Persistent:  False
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    ground_truth:     fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)
    clip_embeddings:  fiftyone.core.fields.VectorField

In [47]:
session = fo.launch_app(dataset_train)